### Nucleotide sequence files

In [9]:

# ============================================================
# FUNCTION: Find nucleotide accession corresponding to protein
# ============================================================

def get_nucleotide_accession(protein_id):
    """
    Search NCBI Protein database for the protein ID and retrieve
    the corresponding nucleotide accession.
    """

    try:
        # Search protein accession
        search = Entrez.esearch(
            db="protein",
            term=f"{protein_id}[Accession]"
        )

        result = Entrez.read(search)
        search.close()

        if not result["IdList"]:
            print(f"  No NCBI protein record found for {protein_id}")
            return None

        protein_ncbi_id = result["IdList"][0]

        # Fetch protein record with linked nucleotide information
        fetch = Entrez.efetch(
            db="protein",
            id=protein_ncbi_id,
            rettype="gb",
            retmode="text"
        )

        record = SeqIO.read(fetch, "genbank")
        fetch.close()

        # Look through CDS cross-references
        for feature in record.features:

            if feature.type == "CDS":

                # Look for coded_by qualifier
                if "coded_by" in feature.qualifiers:

                    coded_by = feature.qualifiers["coded_by"][0]

                    # Example:
                    # NC_000001.11:12345-67890
                    # or
                    # NC_000001.11:complement(12345-67890)

                    nucleotide_accession = coded_by.split(":")[0]

                    return nucleotide_accession

        print(f"  Could not identify nucleotide accession for {protein_id}")

    except Exception as e:
        print(f"  Error searching {protein_id}: {e}")

    return None

In [2]:
# ============================================================
# FUNCTION: Retrieve nucleotide sequence
# ============================================================

def get_nucleotide_sequence(nucleotide_accession):

    try:

        handle = Entrez.efetch(
            db="nuccore",
            id=nucleotide_accession,
            rettype="fasta",
            retmode="text"
        )

        record = SeqIO.read(handle, "fasta")
        handle.close()

        return record

    except Exception as e:

        print(
            f"  Error retrieving nucleotide "
            f"{nucleotide_accession}: {e}"
        )

        return None

In [6]:
import os
os.getcwd()
os.listdir()

['Fasta_match_nt_lociii',
 'aln_lociii_unmatched',
 'find_CPDs_Mary.ipynb',
 'clinvar',
 'clean_up_Mary.ipynb',
 'kondrashov.py',
 '.DS_Store',
 'orthologs_xml_updated.ipynb',
 'aln_lociii_match',
 'Tutorial_Pandas.ipynb',
 'fasta_lociii_rodentia_match',
 'clinvar_lociii.numbers',
 'aln_lociii_rodentia',
 'Outputs_primates',
 'Outputs_rodentia',
 'clinvar_lociii',
 'fasta',
 'pathogenic',
 'fasta_lociii_match_all',
 'fasta_lociii_match',
 'fasta_lociii_match - Copy',
 'fasta_lociii',
 'README.md',
 'orthologs_Rodentia.ipynb',
 'pathogenic_lociii',
 'validation_viz.ipynb',
 'aln_match',
 'Validate_CPDs_Mary.ipynb',
 '.gitignore',
 'fasta_match_Copy',
 'fasta_match',
 'find_closest.ipynb',
 '.github',
 'find_CPDs_Match.ipynb',
 'validation_viz.pdf',
 'orthologs.ipynb',
 'orthologs_xml.ipynb',
 '.ipynb_checkpoints',
 'find_CPDs.ipynb',
 'HBB.numbers',
 'Clinvar_pathogen_Mary.ipynb',
 '.git',
 'orthologs_Mary_Primates.ipynb',
 'aln_lociii_match copy',
 'IQtree',
 'find_closest_Mary.ipynb',

In [4]:
run kondrashov

In [7]:
input_dir = Path("fasta_lociii_match")
output_dir = Path("Fasta_lociii_match_nucleotide")

# Create output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# PROCESS ALL FASTA FILES
# ============================================================

fasta_files = sorted(
    list(input_dir.glob("*.fasta")) +
    list(input_dir.glob("*.fa")) +
    list(input_dir.glob("*.fna"))
)

print(f"Found {len(fasta_files)} FASTA files.\n")


Found 953 FASTA files.



In [8]:
for fasta_file in fasta_files:

    print("=" * 70)
    print(f"Processing: {fasta_file.name}")

    output_file = output_dir / fasta_file.name

    nucleotide_records = []

    # Read protein FASTA
    protein_records = list(SeqIO.parse(fasta_file, "fasta"))

    print(f"Protein records: {len(protein_records)}")


Processing: AAAS_NP_056480.1_match.fasta
Protein records: 33
Processing: ABCA12_NP_775099.2_match.fasta
Protein records: 35
Processing: ABCA1_NP_005493.2_match.fasta
Protein records: 35
Processing: ABCA3_NP_001080.2_match.fasta
Protein records: 32
Processing: ABCA4_NP_000341.2_match.fasta
Protein records: 35
Processing: ABCB11_NP_003733.2_match.fasta
Protein records: 35
Processing: ABCB4_NP_000434.1_match.fasta
Protein records: 35
Processing: ABCC2_NP_000383.2_match.fasta
Protein records: 35
Processing: ABCC6_NP_001162.5_match.fasta
Protein records: 35
Processing: ABCC8_NP_000343.2_match.fasta
Protein records: 34
Processing: ABCC9_NP_001364202.1_match.fasta
Protein records: 29
Processing: ABCD1_NP_000024.2_match.fasta
Protein records: 30
Processing: ACAD9_NP_054768.2_match.fasta
Protein records: 34
Processing: ACADM_NP_000007.1_match.fasta
Protein records: 34
Processing: ACADS_NP_000008.1_match.fasta
Protein records: 34
Processing: ACADVL_NP_000009.1_match.fasta
Protein records: 35
Pro

In [ ]:
from pathlib import Path
from Bio import SeqIO, Entrez, Seq
import time

# ============================================================
# SETTINGS
# ============================================================

input_dir = Path("fasta_lociii_match")
output_dir = Path("Fasta_lociii_match_nucleotide")

# Create output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)



fasta_files = sorted(
    list(input_dir.glob("*.fasta")) +
    list(input_dir.glob("*.fa")) +
    list(input_dir.glob("*.fna"))
)

print(f"Found {len(fasta_files)} FASTA files.\n")


for fasta_file in fasta_files:

    print("=" * 70)
    print(f"Processing: {fasta_file.name}")

    output_file = output_dir / fasta_file.name

    nucleotide_records = []

    # Read protein FASTA
    protein_records = list(SeqIO.parse(fasta_file, "fasta"))

    print(f"Protein records: {len(protein_records)}")

    for protein_record in protein_records:

        # ----------------------------------------------------
        # Extract protein ID
        # ----------------------------------------------------

        protein_id = protein_record.id

        print(f"\nSearching: {protein_id}")

        # ----------------------------------------------------
        # Find corresponding nucleotide accession
        # ----------------------------------------------------

        nucleotide_accession = get_nucleotide_accession(
            protein_id
        )

        if nucleotide_accession is None:
            continue

        print(
            f"  Nucleotide accession: "
            f"{nucleotide_accession}"
        )

        # ----------------------------------------------------
        # Retrieve nucleotide sequence
        # ----------------------------------------------------

        nucleotide_record = get_nucleotide_sequence(
            nucleotide_accession
        )

        if nucleotide_record is None:
            continue

        # ----------------------------------------------------
        # Keep the protein ID in the output
        # ----------------------------------------------------

        nucleotide_record.id = protein_id

        nucleotide_record.description = (
            f"{protein_id} "
            f"[nucleotide={nucleotide_accession}]"
        )

        nucleotide_records.append(nucleotide_record)

        # Avoid hitting NCBI too quickly
        time.sleep(0.34)

    # --------------------------------------------------------
    # Save all nucleotide sequences
    # --------------------------------------------------------

    if nucleotide_records:

        SeqIO.write(
            nucleotide_records,
            output_file,
            "fasta"
        )

        print(
            f"\nSaved {len(nucleotide_records)} sequences → "
            f"{output_file}"
        )

    else:

        print(
            "\nNo nucleotide sequences were retrieved."
        )


print("\n" + "=" * 70)
print("DONE")
print(f"Output directory: {output_dir}")